In [3]:
# WHY BAYESIAN GP (PyMC) > CLASSICAL GP (scikit-learn) > KRIGING
# differ on how they handle hyperparameters

# bayes integrate over posterior (hyperparam uncertianity)

# both ignore hyperparam uncertainity
#opt marginal likelihood
# point estimate variogram

import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

x_obs = np.array([0.0, 1.0, 2.0, 4.0, 5.0])
# sim input data

y_obs = np.array([35.0, 28.0, 42.0, 31.0, 38.0])  # RMR values

X = x_obs[:, None]  # Shape: (5, 1) # pymc gp expect 2d input shape (n, input dim)
y = y_obs            # Shape: (5,)


# define gp model
with pm.Model() as gp_model:
    ell = pm.InverseGamma("ell", alpha=2, beta=2)
    # mean is beta/ (alpha-1)
    # we want ell to be from o to 5 but pushed a bit further from 0 or else overfit

    # signal amplitiude ~ sill
    # to model sigma of 8,  η ∈ [2, 20] is reasonable
    # HalfCauchy is weakly informative — allows large η but regularizes

    eta = pm.HalfCauchy("eta", beta=10)
    #? beta

    # noise, measruement error in rmr (epistemic)

    sigma = pm.HalfNormal("sigma", sigma=5)
    # HalfNormal with σ=5 puts mass on [0, 10]

    # input_dim=1: 1D input (chainage along tunnel)
    # ls=ell: length-scale is our random variable, not a fixed value
    cov_func = eta**2 * pm.gp.cov.Matern52(input_dim=1, ls=ell)

    gp = pm.gp.Marginal(cov_func=cov_func)
    y = gp.marginal_likelihood("y_obs", X=X, y=y, sigma=sigma) # sim y obs and y, confuses

    # this line creating cov matrix, log probab compute and explore via differentiation

#Free parameters: ℓ (length-scale), η (amplitude), σ (noise); priors

#Kernel:           η² × Matérn-5/2(ℓ)")
#Likelihood:       y ~ N(0, K + σ²I)  [zero mean GP]")